# 06pagedata: the data file for the animated map page

**What it does:** gathers everything the page needs into one small JSON file, `docs/data/timemap.json`. The page computes the map positions itself (bearing and travel time from the chosen centre city), so the same file serves the Madrid story and every recentred view.

**Contents of the JSON:**
- `cities`: name, access group in 2025, the train on the fastest 2025 route from Madrid, latitude and longitude.
- `bearing_deg[i][j]`: true bearing from city i to city j; `real_km[i][j]`: true distance (great circle), both as in 04timemap.
- `minutes[era][i][j]`: travel time from 03shortcut.
- `outline`: the outline of mainland Spain on the Madrid map, real and per era (from 05outline), as flat lists x0, y0, x1, y1, ... in km.
- `km_per_min`, the era list, the access group labels and colours, and the data credits.

**Input:** `data/clean/cities.csv`, `data/raw/manual/access_2025.csv`, `data/clean/shortcut_<era>.csv`, `data/clean/timemap_positions.csv`, `data/clean/outline_madrid.csv`.

**Output:** `docs/data/timemap.json`.

**Data credit:** Renfe open GTFS (2025, 2026); FFE Biblioteca Ferroviaria timetable guides (1985, 1993, 2001); hand-compiled backcasts (2010, 2018); Natural Earth (outline, public domain).

## 0. Settings

In [1]:
from pathlib import Path
import json
import numpy as np
import pandas as pd

CLEAN_DIR = Path("data") / "clean"
ACCESS_CSV = Path("data") / "raw" / "manual" / "access_2025.csv"
OUT_DIR = Path("docs") / "data"
OUT_JSON = OUT_DIR / "timemap.json"
ERAS = ["1985", "1993", "2001", "2010", "2018", "2025", "2026"]  # <- eras, in order
STORY_ERA = "2025"  # <- end of the main story (2026 is the slowdown frame)
MAIN_CENTRE = "Madrid"
EARTH_RADIUS_KM = 6371.0
OUTLINE_DECIMALS = 1  # <- km decimals kept for the outline (0.1 km is far below one pixel)
ACCESS_LABEL = {"hub": "Madrid", "full": "high-speed all the way", "part": "high-speed part of the way", "none": "no high-speed"}
ACCESS_COLOUR = {"hub": "#e8e8e8", "full": "#1d6fa5", "part": "#b8860b", "none": "#c2410c"}  # <- VISUAL: validated on light and dark surfaces (hub only for the page)
CREDITS = "Travel times: Renfe open GTFS (2025, 2026); FFE Biblioteca Ferroviaria timetable guides (1985, 1993, 2001); hand-compiled from published times (2010, 2018). Outline: Natural Earth (public domain)."

## 1. Cities, groups and travel times

In [2]:
cities = pd.read_csv(CLEAN_DIR / "cities.csv")
names = list(cities["city"])
access = pd.read_csv(ACCESS_CSV).set_index("city")
times = {}
for era in ERAS:
    table = pd.read_csv(CLEAN_DIR / f"shortcut_{era}.csv", index_col=0).reindex(index=names, columns=names)
    assert table.notna().all().all(), f"missing times in {era}"
    times[era] = table
city_list = []
for name, lat, lon in zip(cities["city"], cities["lat"], cities["lon"]):
    if name == MAIN_CENTRE:
        city_list.append({"name": name, "access": "hub", "train": "", "lat": round(float(lat), 4), "lon": round(float(lon), 4)})
    else:
        city_list.append({"name": name, "access": access.loc[name, "access"], "train": access.loc[name, "train"], "lat": round(float(lat), 4), "lon": round(float(lon), 4)})
print(f"{len(city_list)} cities; access groups: {pd.Series([c['access'] for c in city_list]).value_counts().to_dict()}")

32 cities; access groups: {'full': 21, 'part': 9, 'hub': 1, 'none': 1}


## 2. Bearings and true distances between every pair (as in 04timemap)

In [3]:
lat = np.radians(cities["lat"].to_numpy())
lon = np.radians(cities["lon"].to_numpy())
p0 = lat[:, None]
p1 = lat[None, :]
dl = lon[None, :] - lon[:, None]
bearing = np.degrees(np.arctan2(np.sin(dl) * np.cos(p1), np.cos(p0) * np.sin(p1) - np.sin(p0) * np.cos(p1) * np.cos(dl))) % 360
h = np.sin((p1 - p0) / 2) ** 2 + np.cos(p0) * np.cos(p1) * np.sin(dl / 2) ** 2
real_km = 2 * EARTH_RADIUS_KM * np.arcsin(np.sqrt(h))
np.fill_diagonal(bearing, 0)
print(f"Madrid to Barcelona: bearing {bearing[names.index('Madrid'), names.index('Barcelona')]:.1f} deg, {real_km[names.index('Madrid'), names.index('Barcelona')]:.0f} km (straight line; about 505 km expected)")

Madrid to Barcelona: bearing 75.6 deg, 502 km (straight line; about 505 km expected)


## 3. Scale, and a check against 04timemap

In [4]:
positions = pd.read_csv(CLEAN_DIR / "timemap_positions.csv")
positions["era"] = positions["era"].astype(str)
others = positions[(positions["centre"] == MAIN_CENTRE) & (positions["city"] != MAIN_CENTRE)]
KM_PER_MIN = float((np.hypot(others["x_km"], others["y_km"]) / others["minutes"]).median())
print(f"scale: 1 min = {KM_PER_MIN:.4f} km")

# rebuild every position the way the page will, and compare with 04timemap
worst = 0.0
for era in ERAS:
    m = times[era].to_numpy()
    x = KM_PER_MIN * m * np.sin(np.radians(bearing))
    y = KM_PER_MIN * m * np.cos(np.radians(bearing))
    p = positions[positions["era"] == era]
    i = p["centre"].map(names.index).to_numpy()
    j = p["city"].map(names.index).to_numpy()
    worst = max(worst, float(np.abs(x[i, j] - p["x_km"]).max()), float(np.abs(y[i, j] - p["y_km"]).max()))
print(f"largest difference from timemap_positions.csv: {worst:.3f} km (rounding only if below 0.1)")

scale: 1 min = 0.9939 km
largest difference from timemap_positions.csv: 0.005 km (rounding only if below 0.1)


## 4. Outlines (Madrid view)

In [5]:
outline = pd.read_csv(CLEAN_DIR / "outline_madrid.csv")
outline["era"] = outline["era"].astype(str)
outline_json = {}
counts = set()
for era, part in outline.groupby("era", sort=False):
    part = part.sort_values("point")
    flat = np.column_stack([part["x_km"], part["y_km"]]).round(OUTLINE_DECIMALS).ravel()
    outline_json[era] = [float(v) for v in flat]
    counts.add(len(part))
print(f"outlines: {list(outline_json)}; points per outline: {counts} (must be one number, for the point by point morph)")

outlines: ['real', '1985', '1993', '2001', '2010', '2018', '2025', '2026']; points per outline: {2157} (must be one number, for the point by point morph)


## 5. Write the JSON

In [6]:
data = {
    "eras": ERAS,
    "story_era": STORY_ERA,
    "main_centre": MAIN_CENTRE,
    "km_per_min": round(KM_PER_MIN, 4),
    "access_label": ACCESS_LABEL,
    "access_colour": ACCESS_COLOUR,
    "cities": city_list,
    "bearing_deg": [[round(float(v), 2) for v in row] for row in bearing],
    "real_km": [[round(float(v), 1) for v in row] for row in real_km],
    "minutes": {era: [[int(round(float(v))) for v in row] for row in times[era].to_numpy()] for era in ERAS},
    "outline": outline_json,
    "credits": CREDITS,
}
OUT_DIR.mkdir(parents=True, exist_ok=True)
text = json.dumps(data, allow_nan=False, ensure_ascii=False, separators=(",", ":"))
OUT_JSON.write_text(text, encoding="utf-8")
back = json.loads(OUT_JSON.read_text(encoding="utf-8"))
print(f"saved {OUT_JSON}: {len(text) / 1024:.0f} KB; read back OK: {len(back['cities'])} cities, {len(back['minutes'])} eras, {len(back['outline'])} outlines")

saved docs\data\timemap.json: 256 KB; read back OK: 32 cities, 7 eras, 8 outlines


___
**END**